# 环节 01 · 隔离原语演示

纯 Python 标准库，零依赖。手搓四件事：

1. namespace 位掩码：一组合就是一份"隔离清单"；
2. 五个隔离维度 ↔ 内核原语映射；
3. capability 集合收敛（从"全有"裁到"最小可用"）；
4. seccomp 规则匹配与"默认动作"决策。

> 目的：把"容器隔离"从黑盒拆成可枚举的零件表。

In [ ]:
# §1 namespace 位掩码：CLONE_NEW* 是位标志，可以按需组合
CLONE_NEWNS     = 0x00020000
CLONE_NEWCGROUP = 0x02000000
CLONE_NEWUTS    = 0x04000000
CLONE_NEWIPC    = 0x08000000
CLONE_NEWUSER   = 0x10000000
CLONE_NEWPID    = 0x20000000
CLONE_NEWNET    = 0x40000000

NAMES = {
    "mnt":    CLONE_NEWNS,
    "cgroup": CLONE_NEWCGROUP,
    "uts":    CLONE_NEWUTS,
    "ipc":    CLONE_NEWIPC,
    "user":   CLONE_NEWUSER,
    "pid":    CLONE_NEWPID,
    "net":    CLONE_NEWNET,
}


def show(flags):
    return " | ".join(n for n, f in NAMES.items() if flags & f) or "(none)"


minimal = CLONE_NEWNS | CLONE_NEWPID | CLONE_NEWNET | CLONE_NEWUTS | CLONE_NEWIPC
hardened = minimal | CLONE_NEWUSER | CLONE_NEWCGROUP

print("只开 mount ns      :", show(CLONE_NEWNS))
print("最小容器（缺 user）:", show(minimal))
print("加固容器（含 user）:", show(hardened))
print()
print("缺 user ns 的含义：容器内 root 直接映射到宿主 root（高风险）")

## §2 五个隔离维度 ↔ 原语

namespace 只管"看得见什么"，资源额度是 cgroup 的职责——这是最常见的概念混淆。

In [ ]:
# §2 维度 → 原语映射表
DIMENSIONS = [
    ("文件系统", "mnt ns + overlayfs + pivot_root", "读走 ~/.ssh、改坏宿主文件"),
    ("进程",     "pid ns + capabilities + seccomp",  "kill 别人、ptrace 注入、mount"),
    ("网络",     "net ns + veth + 出站代理",        "外传数据、打内网"),
    ("用户",     "user ns + UID/GID 映射",          "容器内 root ≈ 宿主 root"),
    ("资源",     "cgroup v2（cpu/mem/pids/io）",    "fork 炸弹、吃光内存"),
]

print(f"{'维度':<8}{'原语':<38}{'不挡会怎样'}")
print("-" * 78)
for dim, prim, risk in DIMENSIONS:
    print(f"{dim:<8}{prim:<38}{risk}")

In [ ]:
# §2b 反例：只做 namespace、不做 cgroup，资源维度完全没有限制
def coverage(has_ns, has_cgroup, has_seccomp):
    dims = {
        "文件系统": has_ns, "进程": has_ns, "网络": has_ns,
        "用户": has_ns, "资源": has_cgroup, "内核攻击面": has_seccomp,
    }
    return dims


for label, ns, cg, sc in [
    ("只开 namespace", True, False, False),
    ("namespace + cgroup", True, True, False),
    ("全套（含 seccomp）", True, True, True),
]:
    dims = coverage(ns, cg, sc)
    missing = [k for k, v in dims.items() if not v]
    print(f"{label:<22} 覆盖 {sum(dims.values())}/6  缺口={missing or '无'}")

## §3 capability 收敛：从"全有"裁到"最小可用"

`CAP_SYS_ADMIN` 被称为"新的 root"——它能 mount、改 namespace，等于绕开大半隔离。

In [ ]:
# §3 capability 集合收敛
ALL_CAPS = {
    "CAP_CHOWN", "CAP_NET_ADMIN", "CAP_SYS_ADMIN", "CAP_SYS_PTRACE",
    "CAP_DAC_READ_SEARCH", "CAP_SYS_MODULE", "CAP_SYS_RAWIO",
    "CAP_SETUID", "CAP_SETGID", "CAP_NET_BIND_SERVICE",
}
DANGEROUS = {"CAP_SYS_ADMIN", "CAP_SYS_MODULE", "CAP_SYS_RAWIO",
             "CAP_SYS_PTRACE", "CAP_DAC_READ_SEARCH"}


def harden(caps, needed):
    kept = caps & needed
    dropped = sorted(caps - needed)
    leftover = kept & DANGEROUS
    return kept, dropped, leftover


# 场景 A：跑一个只读分析任务（什么都不需要）
kept, dropped, danger = harden(ALL_CAPS, set())
print("只读分析  保留:", kept or "（空，全丢）", "| 剩余危险项:", danger or "无")

# 场景 B：需要绑 80 端口（NET_BIND_SERVICE 是少数"正常需要"的）
kept, dropped, danger = harden(ALL_CAPS, {"CAP_NET_BIND_SERVICE"})
print("绑低端口  保留:", kept, "| 剩余危险项:", danger or "无")

# 场景 C：图省事保留 CAP_SYS_ADMIN（反模式）
kept, dropped, danger = harden(ALL_CAPS, {"CAP_SYS_ADMIN"})
print("保留 ADMIN:", kept, "| 剩余危险项:", danger, "← 等于没加固")

## §4 seccomp：逐 syscall 的"默认动作 + 规则"

seccomp 只能"允许/拒绝/kill/trap/errno"，**不能改参数、不做路径解析**。

In [ ]:
# §4 seccomp 决策器
DEFAULT_ACTION = "ERRNO(EPERM)"
RULES = {
    "read": "ALLOW", "write": "ALLOW", "openat": "ALLOW",
    "execve": "ALLOW", "socket": "ALLOW",
    "mount": "ERRNO", "ptrace": "ERRNO", "unshare": "ERRNO",
}


def decide(syscall, rules=None, default=DEFAULT_ACTION):
    rules = rules if rules is not None else RULES
    return rules.get(syscall, default)


def run(seq):
    print(f"  默认动作 = {DEFAULT_ACTION}")
    for s in seq:
        act = decide(s)
        flag = "放行" if act == "ALLOW" else "拦截"
        print(f"  {s:<12} → {act:<12} [{flag}]")


print("白名单模式（命中才放行）:")
run(["read", "openat", "execve", "mount", "ptrace"])
print()
print("关键局限：seccomp 拦得住 'openat'，拦不住 '用哪个路径 openat'")
print("→ 路径级管控要靠 LSM / Landlock，不是 seccomp")

In [ ]:
# §4b 收紧白名单能拦掉多少"可用的危险 syscall"
ALL_SYSCALLS = 380
PROFILE = {
    "宽松（只拦几个）": 380 - 5,
    "常见容器默认": 380 - 44,      # Docker 默认 seccomp profile 拦约 44 个
    "严格白名单": 60,               # 只放行真正用到的
}
for name, avail in PROFILE.items():
    print(f"{name:<16} 可用 syscall ≈ {avail:>3} / {ALL_SYSCALLS}"
          f"  （暴露面 {avail / ALL_SYSCALLS:.0%}）")
print()
print("暴露面越小，内核 0day 的可利用入口越少 —— 这是 seccomp 的主要价值")

## §5 创建顺序校验：顺序错了沙箱就搭不起来（或搭错）

In [ ]:
# §5 正确的创建序列（顺序敏感）
CORRECT = [
    "clone(namespaces)",   # 1 先建视图
    "mount / 私有化",       # 2 禁止挂载外泄
    "overlayfs 只读根",     # 3
    "pivot_root + 摘旧根",  # 4
    "cgroup 配额",          # 5
    "setgroups/setuid 降权",  # 6
    "capset drop",          # 7 必须在 setuid 之后
    "no_new_privs",         # 8 必须在 seccomp 之前
    "seccomp 白名单",       # 9 最后上
    "execve 目标命令",      # 10
]
INDEX = {step: i for i, step in enumerate(CORRECT)}


def check(order):
    problems = []
    for i in range(len(order) - 1):
        a, b = order[i], order[i + 1]
        if b not in INDEX or a not in INDEX:
            problems.append(f"未知步骤 {a!r}/{b!r}")
            continue
        if INDEX[b] < INDEX[a]:
            problems.append(f"{b} 早于 {a}")
    return problems


print("正确顺序     →", check(CORRECT) or "OK")
bad = list(CORRECT)
bad[6], bad[5] = bad[5], bad[6]   # capset 提前到 setuid 之前
print("capset 提前  →", check(bad))
bad2 = list(CORRECT)
bad2[8], bad2[7] = bad2[7], bad2[8]  # seccomp 提前到 no_new_privs 之前
print("seccomp 提前 →", check(bad2))

## §6 自测表

| # | 问题 | 答案要点 |
|---|---|---|
| 1 | namespace 能限制 CPU/内存吗？ | 不能；那是 cgroup 的职责 |
| 2 | 为什么 `CAP_SYS_ADMIN` 危险？ | 能 mount / 改 ns / 大量 ioctl，等价绕过隔离 |
| 3 | seccomp 能限制"只能读某目录"吗？ | 不能；只做 syscall 级允许/拒绝 |
| 4 | `no_new_privs` 解决什么？ | 阻断 setuid 程序涨权限、防 seccomp 被重置 |
| 5 | `chroot` 为什么不够？ | 无 pid/net/user ns、无 cap 裁剪、无 cgroup |
| 6 | user namespace 的核心价值？ | 容器内 root 只映射到宿主非特权 UID |
| 7 | `pids.max` 触顶报什么错？ | `fork()` 返回 `EAGAIN`，不是权限错误 |

**相关长文**：[环节01-隔离原语详解.md](./环节01-隔离原语详解.md)